# Urai QLoRA training — free GPU (Kaggle 30h/wk T4x2 recommended, Colab T4 works)
**Kaggle:** Settings → Accelerator → GPU T4 x2, Internet → ON. **Colab:** Runtime → T4 GPU.
Trains Qwen3-0.6B (default, ~15 min) or 1.7B on Urai's verified distill set → LoRA adapter → push to HF Hub.
Bring the adapter home: merge + GGUF export, then `urai model bench` + registry. Full loop: `docs/TRAINING.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'NO GPU - enable it (see top cell)'
import torch; print('cuda:', torch.cuda.is_available(), '| gpus:', torch.cuda.device_count())

In [ ]:
# Unsloth (2x faster, 70% less VRAM) + stack. Ref: https://github.com/unslothai/unsloth
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl peft accelerate bitsandbytes datasets 2>&1 | tail -n 2

In [ ]:
# Get the verified training set (first found wins):
#  1. Kaggle dataset attached at /kaggle/input/<name>/*.jsonl
#  2. file uploaded into the notebook (Colab files.upload / Kaggle input)
#  3. HF Hub dataset (needs token for private)
import glob, os
CANDS = sorted(glob.glob('/kaggle/input/*/*.jsonl')) + sorted(glob.glob('*.jsonl'))
TRAIN_FILE = os.environ.get('URAI_TRAIN_FILE') or (CANDS[0] if CANDS else '')
print('using:', TRAIN_FILE or 'NOTHING FOUND - upload distill-v0.2.jsonl (see pack_dataset.py)')

In [ ]:
import json
ALPACA = ('Below is an instruction. Write a response.\n\n### Instruction:\n{instruction}\n\n### Response:\n{output}')
rows = []
for line in open(TRAIN_FILE):
    line = line.strip()
    if not line: continue
    r = json.loads(line)
    t, c = r.get('task', {}), r.get('completion', '')
    p = t.get('prompt', '') if isinstance(t, dict) else str(t)
    if p and c: rows.append({'text': ALPACA.format(instruction=p, output=c)})
print('training rows:', len(rows))
assert rows, 'empty training set'
from datasets import Dataset
ds = Dataset.from_list(rows)

In [ ]:
# QLoRA: 4-bit base, r=16, all linear layers (canonical recipe, docs/TRAINING.md)
MODEL = 'unsloth/Qwen3-0.6B-bnb-4bit'  # or: unsloth/Qwen3-1.7B-bnb-4bit (needs 16GB+)
OUT = 'urai-qwen3-06b-v1'
try:
    from unsloth import FastModel as Fast
except ImportError:
    from unsloth import FastLanguageModel as Fast
model, tok = Fast.from_pretrained(model_name=MODEL, max_seq_length=2048, dtype=None, load_in_4bit=True)
model = Fast.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=3407)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=TrainingArguments(
    output_dir=OUT, per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=1, learning_rate=2e-4, logging_steps=5, save_steps=200,
    optim='adamw_8bit', seed=3407, report_to='none')).train()
model.save_pretrained(OUT); tok.save_pretrained(OUT)
print('adapter saved ->', OUT)

In [ ]:
# Push adapter to HF Hub (free, versioned home for Urai weights). Needs a write token:
#   from huggingface_hub import login; login('hf_...')   # or set HF_TOKEN env
REPO = 'tukitattu/urai-qwen3-06b-v1'  # change me
model.push_to_hub(REPO); tok.push_to_hub(REPO)
print('pushed:', REPO)
# Back home: merge + GGUF, e.g.
#   model.save_pretrained_merged(OUT+'-merged', tok, save_method='merged_16bit')
#   model.save_pretrained_gguf(OUT+'-gguf', tok, quantization_method='q4_k_m')
# then: urai model bench + register in model/registry.json (docs/TRAINING.md)